# Sepsis Early-Warning — Model Training

Trains an XGBoost classifier to predict sepsis onset from hourly ICU vitals/labs, using
the patient-level train/test split created during preprocessing.

**Prerequisite:** run `notebooks/02_sepsis_preprocessing.ipynb` first so
`data/sepsis_train.csv` and `data/sepsis_test.csv` exist.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import (roc_auc_score, roc_curve, precision_recall_curve, auc,
                               classification_report, confusion_matrix, ConfusionMatrixDisplay)
import xgboost as xgb
import joblib
import os

os.makedirs("models", exist_ok=True)

train_df = pd.read_csv("data/sepsis_train.csv")
test_df = pd.read_csv("data/sepsis_test.csv")
print(f"Train: {train_df.shape}, Test: {test_df.shape}")

## 1. Select features

We exclude identifiers (`PatientID`, `HospitalSystem`) and the label itself
(`SepsisLabel`) from the feature set — everything else feeds the model.

In [ ]:
drop_cols = ["PatientID", "HospitalSystem", "SepsisLabel"]
feature_cols = [c for c in train_df.columns if c not in drop_cols]

X_train, y_train = train_df[feature_cols], train_df["SepsisLabel"]
X_test, y_test = test_df[feature_cols], test_df["SepsisLabel"]

print(f"{len(feature_cols)} features")
print(f"Train sepsis rate: {y_train.mean():.4f}")
print(f"Test sepsis rate:  {y_test.mean():.4f}")

## 2. Train XGBoost

Sepsis-positive hours are rare (~1.8%) — again we use `scale_pos_weight` to counter the
imbalance. `tree_method="hist"` speeds up training substantially on this larger dataset
(1.2M+ training rows) with no meaningful accuracy trade-off.

In [ ]:
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
print(f"scale_pos_weight: {scale_pos_weight:.2f}")

model = xgb.XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.05,
    scale_pos_weight=scale_pos_weight,
    eval_metric="auc",
    tree_method="hist",
    random_state=42,
)
model.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)
print("Training complete.")

## 3. Evaluate

Same reasoning as the readmission model: with ~1.8% positive rate, AUROC and AUPRC are
the metrics that actually reflect model quality — accuracy alone would be misleading.

In [ ]:
y_pred_proba = model.predict_proba(X_test)[:, 1]
y_pred = (y_pred_proba > 0.5).astype(int)

auroc = roc_auc_score(y_test, y_pred_proba)
precision, recall, _ = precision_recall_curve(y_test, y_pred_proba)
auprc = auc(recall, precision)

print(f"AUROC: {auroc:.4f}")
print(f"AUPRC: {auprc:.4f}")
print()
print(classification_report(y_test, y_pred, target_names=["No sepsis", "Sepsis"]))

## 4. ROC and Precision-Recall curves

In [ ]:
fpr, tpr, _ = roc_curve(y_test, y_pred_proba)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].plot(fpr, tpr, label=f"AUROC = {auroc:.3f}", color="#2563eb", linewidth=2)
axes[0].plot([0, 1], [0, 1], linestyle="--", color="gray", label="Random")
axes[0].set_xlabel("False Positive Rate")
axes[0].set_ylabel("True Positive Rate")
axes[0].set_title("ROC Curve — Sepsis Early-Warning Model")
axes[0].legend()

axes[1].plot(recall, precision, label=f"AUPRC = {auprc:.3f}", color="#dc2626", linewidth=2)
axes[1].axhline(y=y_test.mean(), linestyle="--", color="gray", label="Baseline (prevalence)")
axes[1].set_xlabel("Recall")
axes[1].set_ylabel("Precision")
axes[1].set_title("Precision-Recall Curve — Sepsis Early-Warning Model")
axes[1].legend()

plt.tight_layout()
plt.savefig("models/sepsis_roc_pr_curves.png", dpi=150, bbox_inches="tight")
plt.show()

## 5. Confusion matrix

In [ ]:
cm = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(6, 5))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["No sepsis", "Sepsis"])
disp.plot(ax=ax, cmap="Blues", colorbar=False)
ax.set_title("Confusion Matrix — Sepsis Model (threshold = 0.5)")
plt.tight_layout()
plt.savefig("models/sepsis_confusion_matrix.png", dpi=150, bbox_inches="tight")
plt.show()

## 6. Feature importance

Worth noting which features the model actually relies on — this is a good sanity check
(do the top features make clinical sense?) and sets up the explainability work we'll do
next with SHAP.

In [ ]:
importances = pd.Series(model.feature_importances_, index=feature_cols).sort_values(ascending=False).head(15)

fig, ax = plt.subplots(figsize=(8, 6))
importances.sort_values().plot(kind="barh", ax=ax, color="#2563eb")
ax.set_title("Top 15 Feature Importances — Sepsis Model")
ax.set_xlabel("Importance")
plt.tight_layout()
plt.savefig("models/sepsis_feature_importance.png", dpi=150, bbox_inches="tight")
plt.show()

## 7. Save the model

In [ ]:
joblib.dump(model, "models/sepsis_model.pkl")
joblib.dump(feature_cols, "models/sepsis_feature_names.pkl")
print("Saved model and feature names to models/")